# Online Retail — Câu 12: FP-Growth và so sánh Apriori

**Võ Thiện Linh — MSSV: 6451071041**

Tìm tập phổ biến bằng FP-Growth với `min_sup = 3%`, tạo luật với `min_conf = 50%`, rồi so sánh tập phổ biến và luật với kết quả Apriori. Hai thuật toán được cài đặt trực tiếp bằng Python, không cần cài thêm thư viện khai phá luật kết hợp.

In [ ]:
print('Võ Thiện Linh - MSSV: 6451071041')
from pathlib import Path
from collections import Counter
from itertools import combinations
import math
import pandas as pd
from IPython.display import display
candidates = [Path('Online Retail.xlsx'), Path('LAB1') / 'Phan02' / '6451071041_LAB1_Bai2_Trang37' / 'Online Retail.xlsx', Path('/content/Online Retail.xlsx')]
data_path = next((p for p in candidates if p.is_file()), None)
if data_path is None:
    raise FileNotFoundError(f'Không tìm thấy Online Retail.xlsx. Thư mục đang chạy: {Path.cwd()}')
df = pd.read_excel(data_path, engine='openpyxl')
df['Description'] = df['Description'].str.strip()
df = df.dropna(subset=['InvoiceNo']).copy()
df['InvoiceNo'] = df['InvoiceNo'].astype('string').str.strip()
df = df.loc[~df['InvoiceNo'].str.contains('C', regex=False, na=False)]
uk = df.loc[df['Country'].eq('United Kingdom')]
grouped = uk.groupby(['InvoiceNo', 'Description'], as_index=False, dropna=True)['Quantity'].sum()
invoice_ids = pd.Index(grouped['InvoiceNo'].drop_duplicates(), name='InvoiceNo')
purchases = grouped.loc[grouped['Quantity'].gt(0) & grouped['Description'].ne('POSTAGE')]
items_by_invoice = purchases.groupby('InvoiceNo')['Description'].agg(frozenset).to_dict()
transactions = [items_by_invoice.get(invoice, frozenset()) for invoice in invoice_ids]
def apriori_counts(transactions, min_support):
    minimum_count = math.ceil(min_support * len(transactions))
    item_counts = Counter(item for transaction in transactions for item in transaction)
    previous = {frozenset([item]): count for item, count in item_counts.items() if count >= minimum_count}
    frequent_counts = dict(previous)
    frequent_items = set(item_counts)
    size = 2
    while previous:
        previous_list = sorted(previous, key=lambda itemset: tuple(sorted(itemset)))
        previous_set = set(previous)
        candidates = set()
        for i, left in enumerate(previous_list):
            for right in previous_list[i + 1:]:
                candidate = left | right
                if len(candidate) == size and all(frozenset(part) in previous_set for part in combinations(candidate, size - 1)):
                    candidates.add(candidate)
        if not candidates:
            break
        candidate_counts = Counter()
        for transaction in transactions:
            active_items = sorted(transaction & frequent_items)
            for combination in combinations(active_items, size):
                itemset = frozenset(combination)
                if itemset in candidates:
                    candidate_counts[itemset] += 1
        previous = {itemset: count for itemset, count in candidate_counts.items() if count >= minimum_count}
        frequent_counts.update(previous)
        size += 1
    return frequent_counts, minimum_count
class FPNode:
    def __init__(self, item, parent=None):
        self.item = item
        self.count = 0
        self.parent = parent
        self.children = {}

def build_fp_tree(weighted_transactions, minimum_count):
    item_counts = Counter()
    for items, weight in weighted_transactions:
        item_counts.update({item: weight for item in set(items)})
    frequent = {item: count for item, count in item_counts.items() if count >= minimum_count}
    if not frequent:
        return None, None, None
    root = FPNode(None)
    headers = {item: [] for item in frequent}
    for items, weight in weighted_transactions:
        ordered = sorted((item for item in set(items) if item in frequent), key=lambda item: (-frequent[item], item))
        current = root
        for item in ordered:
            child = current.children.get(item)
            if child is None:
                child = FPNode(item, current)
                current.children[item] = child
                headers[item].append(child)
            child.count += weight
            current = child
    return root, headers, frequent

def fp_growth(weighted_transactions, minimum_count, prefix=frozenset()):
    root, headers, frequent = build_fp_tree(weighted_transactions, minimum_count)
    if root is None:
        return {}
    frequent_counts = {}
    for item in sorted(headers, key=lambda value: (frequent[value], value)):
        pattern = prefix | frozenset([item])
        frequent_counts[pattern] = frequent[item]
        conditional_paths = []
        for node in headers[item]:
            path = []
            parent = node.parent
            while parent is not None and parent.item is not None:
                path.append(parent.item)
                parent = parent.parent
            if path:
                conditional_paths.append((path, node.count))
        if conditional_paths:
            frequent_counts.update(fp_growth(conditional_paths, minimum_count, pattern))
    return frequent_counts
def make_rules(itemset_counts, transaction_count, min_confidence):
    rules = []
    for itemset, itemset_count in itemset_counts.items():
        if len(itemset) < 2:
            continue
        for antecedent_size in range(1, len(itemset)):
            for left in combinations(sorted(itemset), antecedent_size):
                antecedent = frozenset(left)
                consequent = itemset - antecedent
                confidence = itemset_count / itemset_counts[antecedent]
                if confidence >= min_confidence:
                    support = itemset_count / transaction_count
                    consequent_support = itemset_counts[consequent] / transaction_count
                    rules.append({'antecedent': antecedent, 'consequent': consequent, 'support': support, 'confidence': confidence, 'lift': confidence / consequent_support})
    return rules
min_support = 0.03
min_confidence = 0.50
minimum_count = math.ceil(min_support * len(transactions))
apriori_frequent_counts, _ = apriori_counts(transactions, min_support)
fp_frequent_counts = fp_growth([(list(transaction), 1) for transaction in transactions], minimum_count)
apriori_rules = make_rules(apriori_frequent_counts, len(transactions), min_confidence)
fp_rules = make_rules(fp_frequent_counts, len(transactions), min_confidence)
def rules_as_keys(rules):
    return {(rule['antecedent'], rule['consequent'], rule['support'], rule['confidence']) for rule in rules}

itemsets_match = apriori_frequent_counts == fp_frequent_counts
rules_match = rules_as_keys(apriori_rules) == rules_as_keys(fp_rules)
comparison = pd.DataFrame([
    {'algorithm': 'Apriori', 'frequent_itemsets': len(apriori_frequent_counts), 'association_rules': len(apriori_rules)},
    {'algorithm': 'FP-Growth', 'frequent_itemsets': len(fp_frequent_counts), 'association_rules': len(fp_rules)}
])
print(f'Giao dịch: {len(transactions):,}; min_sup = {min_support:.0%} ({minimum_count:,} hóa đơn); min_conf = {min_confidence:.0%}')
print('Tập phổ biến Apriori và FP-Growth trùng nhau:', itemsets_match)
print('Luật kết hợp Apriori và FP-Growth trùng nhau:', rules_match)
display(comparison)
fp_itemsets = pd.DataFrame([{'itemsets': ', '.join(sorted(itemset)), 'itemset_size': len(itemset), 'support': count / len(transactions), 'invoice_count': count} for itemset, count in fp_frequent_counts.items()]).sort_values(['itemset_size', 'support'], ascending=[True, False]).reset_index(drop=True)
fp_rules_df = pd.DataFrame([{'antecedent': ', '.join(sorted(rule['antecedent'])), 'consequent': ', '.join(sorted(rule['consequent'])), 'support': rule['support'], 'confidence': rule['confidence'], 'lift': rule['lift']} for rule in fp_rules])
if not fp_rules_df.empty:
    fp_rules_df = fp_rules_df.sort_values(['confidence', 'support'], ascending=False).reset_index(drop=True)
print('Tập phổ biến tìm được bằng FP-Growth:')
display(fp_itemsets)
print('Luật kết hợp tìm được bằng FP-Growth:')
display(fp_rules_df)
assert itemsets_match and rules_match